# Module 00: The Feature Selection Landscape — Overview

## Learning Objectives

By completing this notebook, you will:
1. Observe how model performance varies as feature count changes — and see the Hughes phenomenon in real data
2. Visualise distance concentration in high dimensions with a concrete simulation
3. Measure and compare the wall-clock time of filter, wrapper, and embedded selection on the same dataset
4. Build the intuition that guides method choice for the rest of the course

## Prerequisites
- Python with scikit-learn, NumPy, Matplotlib, and pandas installed
- Familiarity with cross-validated model evaluation

## Estimated Time: 15 minutes

## Dataset

We use the **Madelon** dataset (OpenML ID 1485), a synthetic but challenging classification problem designed specifically to study feature selection. It has 500 features, 2,600 samples, and only 20 features are informative — the other 480 are noise. It is the canonical benchmark for feature selection research.

In [ ]:
# Standard imports — all available in a standard data science environment
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.datasets import fetch_openml, make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression, Lasso
from sklearn.feature_selection import (
    SelectKBest, f_classif, mutual_info_classif,
    RFE, SelectFromModel
)
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')
np.random.seed(42)

# Plotting defaults for clear, readable figures
plt.rcParams.update({
    'figure.dpi': 100,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

print('Imports complete.')

## Section 1: Load the Madelon Dataset

Madelon was created for the 2003 NIPS Feature Selection Challenge. Key properties:
- **500 features** (20 informative, 480 noise)
- **2,600 samples**
- **Binary classification** task
- $p/n$ ratio = 0.19 — in the moderate-risk zone from our taxonomy guide

This is a realistic dimensionality ratio for many industry applications.

In [ ]:
def load_madelon():
    """Load the Madelon dataset from OpenML, with a fallback to local generation.
    
    Returns
    -------
    X : ndarray of shape (n_samples, 500)
    y : ndarray of shape (n_samples,)
    """
    try:
        print('Fetching Madelon from OpenML... (first run downloads ~3MB)')
        dataset = fetch_openml(name='madelon', version=1, as_frame=False,
                               parser='auto')
        X = dataset.data.astype(np.float64)
        y = (dataset.target == dataset.target[0]).astype(int)  # encode as 0/1
        print(f'Loaded from OpenML: {X.shape[0]} samples, {X.shape[1]} features')
    except Exception as exc:
        print(f'OpenML fetch failed ({exc}). Generating equivalent synthetic dataset.')
        # The synthetic version preserves Madelon's key properties:
        # 500 features, 20 informative, binary classification
        X, y = make_classification(
            n_samples=2600, n_features=500, n_informative=20,
            n_redundant=10, n_repeated=0, n_clusters_per_class=2,
            random_state=42
        )
        print(f'Generated synthetic Madelon: {X.shape[0]} samples, {X.shape[1]} features')
    return X, y


X, y = load_madelon()

print(f'\nDataset summary:')
print(f'  Samples (n): {X.shape[0]}')
print(f'  Features (p): {X.shape[1]}')
print(f'  p/n ratio: {X.shape[1]/X.shape[0]:.3f}')
print(f'  Class balance: {np.mean(y==0):.1%} class 0, {np.mean(y==1):.1%} class 1')
print(f'  X dtype: {X.dtype}')

# Standardise features — important for Lasso and distance calculations
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## Section 2: Performance vs. Feature Count — The Hughes Phenomenon

The taxonomy guide described the Hughes phenomenon: classifier accuracy can *decrease* as you add features beyond the optimal count. We will now observe this directly.

**Method:** We select the top-$k$ features using a univariate F-statistic filter, then train a logistic regression and measure 5-fold cross-validated accuracy. We repeat for $k = 5, 10, 20, 50, 100, 200, 500$.

Because the F-statistic filter ranks features by their individual association with the target, the top-$k$ features for small $k$ are the most informative ones. As $k$ grows, we start including noise features — and performance eventually drops.

In [ ]:
def evaluate_top_k_features(
    X, y, k_values, model, scorer='accuracy', cv=5, random_state=42
):
    """Evaluate cross-validated performance for the top-k features by F-statistic.
    
    Parameters
    ----------
    X : array of shape (n_samples, n_features) — standardised
    y : array of shape (n_samples,)
    k_values : list of int — feature counts to evaluate
    model : sklearn estimator — trained fresh at each k
    scorer : str — metric passed to cross_val_score
    cv : int — number of cross-validation folds
    random_state : int
    
    Returns
    -------
    results : dict with keys 'k', 'mean_score', 'std_score'
    """
    cv_splitter = StratifiedKFold(n_splits=cv, shuffle=True,
                                  random_state=random_state)
    mean_scores = []
    std_scores = []

    for k in k_values:
        # Select top-k features by F-statistic (fast, model-free)
        selector = SelectKBest(f_classif, k=k)
        X_k = selector.fit_transform(X, y)

        # Cross-validate the model on the selected features
        scores = cross_val_score(model, X_k, y, cv=cv_splitter,
                                  scoring=scorer, n_jobs=-1)
        mean_scores.append(scores.mean())
        std_scores.append(scores.std())

    return {'k': k_values, 'mean_score': mean_scores, 'std_score': std_scores}


# Feature counts to evaluate — spanning 1% to 100% of all features
k_values = [5, 10, 15, 20, 30, 50, 75, 100, 150, 200, 300, 500]

print('Evaluating logistic regression across feature counts...')
print('(This runs 5-fold CV for each k — about 30 seconds total)')
t0 = time.time()

lr_model = LogisticRegression(max_iter=300, random_state=42)
lr_results = evaluate_top_k_features(X_scaled, y, k_values, lr_model)

elapsed = time.time() - t0
print(f'Done in {elapsed:.1f} seconds.')

# Print the table — easy to see the pattern
print(f'\n{"k":>6} {"Mean Accuracy":>14} {"Std":>6}')
print('-' * 30)
for k, mean, std in zip(
    lr_results['k'], lr_results['mean_score'], lr_results['std_score']
):
    print(f'{k:>6}  {mean:.4f}           {std:.4f}')

In [ ]:
# Visualise the performance curve — the Hughes phenomenon should be visible
fig, ax = plt.subplots(figsize=(9, 5))

k_arr = np.array(lr_results['k'])
mean_arr = np.array(lr_results['mean_score'])
std_arr = np.array(lr_results['std_score'])

ax.plot(k_arr, mean_arr, color='steelblue', linewidth=2.5, marker='o',
        markersize=6, label='Mean CV accuracy')
ax.fill_between(
    k_arr,
    mean_arr - std_arr,
    mean_arr + std_arr,
    alpha=0.2, color='steelblue', label='±1 std'
)

# Mark the peak — optimal feature count
peak_idx = np.argmax(mean_arr)
ax.axvline(k_arr[peak_idx], color='crimson', linestyle='--', linewidth=1.5,
           label=f'Peak at k={k_arr[peak_idx]} ({mean_arr[peak_idx]:.3f})')

# Mark the theoretical number of informative features in Madelon
ax.axvline(20, color='darkgreen', linestyle=':', linewidth=1.5,
           label='Madelon informative features (20)')

ax.set_xlabel('Number of features selected (k)', fontsize=12)
ax.set_ylabel('5-fold CV accuracy', fontsize=12)
ax.set_title('Hughes Phenomenon: Accuracy vs. Feature Count\n'
             'Madelon dataset, Logistic Regression', fontsize=13)
ax.legend(fontsize=10)
ax.set_xscale('log')
ax.set_xticks(k_values)
ax.set_xticklabels(k_values, rotation=45)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('hughes_phenomenon.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'\nPeak accuracy {mean_arr[peak_idx]:.4f} at k={k_arr[peak_idx]} features')
print(f'Accuracy at all 500 features: {mean_arr[-1]:.4f}')
print(f'Improvement from selection: +{(mean_arr[peak_idx] - mean_arr[-1]):.4f}')

**What you should see:** Accuracy rises as we add informative features, peaks somewhere near $k = 20$–$50$, then falls as noise features dilute the signal. This is the Hughes phenomenon in real data — more features is *not* always better.

The exact peak location depends on the filter quality (F-statistic does not perfectly rank all informative features first) and the regularisation of logistic regression. The key observation is the eventual decline.

## Section 3: The Curse of Dimensionality — Distance Concentration

The dimensionality guide showed mathematically that pairwise distances become increasingly similar as $d$ grows. Here we simulate this directly.

**Method:** Generate $n = 500$ points uniformly from $[0,1]^d$ for increasing $d$. For each $d$, compute all pairwise Euclidean distances and plot the distribution. Watch how the distribution collapses from wide (meaningful distance variation) to narrow (all distances similar) as $d$ increases.

In [ ]:
from sklearn.metrics import pairwise_distances


def compute_distance_stats(n_points: int, n_dims: int, n_samples: int = 10) -> dict:
    """Compute pairwise distance statistics for uniform points in [0,1]^d.
    
    Parameters
    ----------
    n_points : int — number of points to generate
    n_dims : int — dimensionality d
    n_samples : int — number of random replicates (for stable estimates)
    
    Returns
    -------
    dict with 'mean_dist', 'std_dist', 'cv' (coefficient of variation),
    'relative_contrast' = (max - min) / min
    """
    cvs = []
    contrasts = []
    mean_dists = []

    for _ in range(n_samples):
        # Uniform random points in [0,1]^d
        X = np.random.uniform(0, 1, size=(n_points, n_dims))
        # Pairwise Euclidean distances (upper triangle only)
        D = pairwise_distances(X, metric='euclidean')
        upper = D[np.triu_indices_from(D, k=1)]

        cvs.append(upper.std() / upper.mean())
        contrasts.append((upper.max() - upper.min()) / upper.min())
        mean_dists.append(upper.mean())

    return {
        'mean_dist': np.mean(mean_dists),
        'std_dist': np.mean([np.std(np.random.uniform(0,1,(n_points,n_dims))
                              .reshape(n_points,-1)) for _ in range(3)]),
        'cv': np.mean(cvs),
        'relative_contrast': np.mean(contrasts),
    }


# Evaluate at increasing dimensionalities
dim_values = [1, 2, 5, 10, 20, 50, 100, 200, 500]
n_points = 300  # keep manageable for fast execution

stats_by_dim = {}
print('Computing distance statistics across dimensions...')
for d in dim_values:
    stats_by_dim[d] = compute_distance_stats(n_points, d, n_samples=5)
    print(f'  d={d:4d}: mean_dist={stats_by_dim[d]["mean_dist"]:.3f}, '
          f'CV={stats_by_dim[d]["cv"]:.4f}, '
          f'relative_contrast={stats_by_dim[d]["relative_contrast"]:.3f}')

print('Done.')

In [ ]:
# Plot the distance distribution at 4 representative dimensions
# The collapsing width of the distribution is the curse of dimensionality made visual
selected_dims = [2, 10, 50, 200]

fig, axes = plt.subplots(1, 4, figsize=(14, 4), sharey=False)

colours = ['#2196F3', '#FF9800', '#E91E63', '#4CAF50']

for ax, d, colour in zip(axes, selected_dims, colours):
    X_sim = np.random.uniform(0, 1, size=(n_points, d))
    D = pairwise_distances(X_sim, metric='euclidean')
    distances = D[np.triu_indices_from(D, k=1)]

    ax.hist(distances, bins=50, color=colour, alpha=0.8, density=True,
            edgecolor='none')
    ax.set_title(f'd = {d}\nCV = {distances.std()/distances.mean():.3f}',
                 fontsize=12)
    ax.set_xlabel('Pairwise distance', fontsize=10)
    ax.set_ylabel('Density' if ax == axes[0] else '', fontsize=10)
    ax.axvline(distances.mean(), color='black', linestyle='--',
               linewidth=1.2, label=f'Mean={distances.mean():.2f}')
    ax.legend(fontsize=8)

fig.suptitle(
    'Distance Concentration: Pairwise Distance Distributions\n'
    'as Dimensionality Grows (n=300 uniform points per panel)',
    fontsize=13, y=1.02
)
plt.tight_layout()
plt.savefig('distance_concentration.png', dpi=120, bbox_inches='tight')
plt.show()

# Show the CV collapse as a summary table
print('\nCoefficient of Variation (CV) of pairwise distances by dimension:')
print(f'{"d":>6}  {"Mean dist":>10}  {"CV":>8}  {"Rel. contrast":>14}')
print('-' * 45)
for d in dim_values:
    s = stats_by_dim[d]
    print(f'{d:>6}  {s["mean_dist"]:>10.4f}  {s["cv"]:>8.4f}  '
          f'{s["relative_contrast"]:>14.4f}')

**What you should see:** As $d$ increases, the distance histogram narrows — the coefficient of variation (CV = std/mean) shrinks toward zero. At $d=2$, distances vary widely. At $d=200$, nearly all pairwise distances are nearly identical. This is why KNN, kernel methods, and density estimators fail in high dimensions.

The relative contrast $(d_\text{max} - d_\text{min})/d_\text{min}$ also collapses — confirming the Beyer et al. (1999) result.

## Section 4: Timing Filter vs. Wrapper vs. Embedded

The complexity guide gave theoretical formulas. Here we measure actual wall-clock times for all three families on the Madelon dataset.

**Methods compared:**
- **Filter:** SelectKBest with F-statistic ($k=20$) — $O(p \cdot n)$
- **Wrapper:** RFE with logistic regression ($k=20$) — $O(p \cdot T_m)$
- **Embedded:** Logistic Regression with L1 penalty (select non-zero coefficients) — $O(T_m)$

In [ ]:
def time_selection_method(method_fn, n_repeats: int = 3) -> tuple[float, int]:
    """Run a selection function n_repeats times and return (mean_seconds, n_features).
    
    Parameters
    ----------
    method_fn : callable — takes no arguments, returns (X_selected, n_features)
    n_repeats : int — number of timed runs for stable measurement
    
    Returns
    -------
    mean_time : float — mean elapsed seconds across runs
    n_selected : int — number of features selected
    """
    times = []
    n_selected = None
    for _ in range(n_repeats):
        t0 = time.perf_counter()
        _, n_sel = method_fn()
        times.append(time.perf_counter() - t0)
        n_selected = n_sel
    return float(np.mean(times)), n_selected


# ── Filter method: F-statistic SelectKBest ────────────────────────────────────
def filter_method():
    selector = SelectKBest(f_classif, k=20)
    X_sel = selector.fit_transform(X_scaled, y)
    return X_sel, X_sel.shape[1]


# ── Wrapper method: RFE with logistic regression ──────────────────────────────
# step=10 removes 10 features per iteration to speed up the demo
def wrapper_method():
    estimator = LogisticRegression(max_iter=200, random_state=42)
    selector = RFE(estimator, n_features_to_select=20, step=10)
    X_sel = selector.fit_transform(X_scaled, y)
    return X_sel, X_sel.shape[1]


# ── Embedded method: L1-penalised logistic regression ────────────────────────
def embedded_method():
    # Train L1 logistic regression — coefficients of zero = excluded features
    lr_l1 = LogisticRegression(
        penalty='l1', solver='liblinear', C=0.1, max_iter=300, random_state=42
    )
    selector = SelectFromModel(lr_l1, threshold=1e-5)
    X_sel = selector.fit_transform(X_scaled, y)
    return X_sel, X_sel.shape[1]


methods = {
    'Filter (F-statistic, k=20)': filter_method,
    'Wrapper (RFE, step=10, k=20)': wrapper_method,
    'Embedded (L1 Logistic, C=0.1)': embedded_method,
}

print(f'Timing feature selection methods on Madelon (p={X.shape[1]}, n={X.shape[0]})')
print('=' * 65)

timing_results = {}
for method_name, method_fn in methods.items():
    print(f'  Running: {method_name}...', end=' ', flush=True)
    mean_time, n_sel = time_selection_method(method_fn, n_repeats=3)
    timing_results[method_name] = {'time': mean_time, 'n_selected': n_sel}
    print(f'done — {mean_time:.3f}s ({n_sel} features selected)')

print('\nSummary:')
filter_time = timing_results['Filter (F-statistic, k=20)']['time']
for name, result in timing_results.items():
    ratio = result['time'] / filter_time
    print(f'  {name}')
    print(f'    Time: {result["time"]:.3f}s  |  '
          f'Ratio vs filter: {ratio:.1f}x  |  '
          f'Features selected: {result["n_selected"]}')

In [ ]:
# Bar chart comparing wall-clock times across methods
method_labels = list(timing_results.keys())
times = [timing_results[m]['time'] for m in method_labels]
colours_bar = ['#4CAF50', '#FF9800', '#2196F3']  # Green=fast, Orange=medium, Blue=embedded

fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(
    range(len(method_labels)), times,
    color=colours_bar, alpha=0.85, edgecolor='white', linewidth=1.5
)

# Annotate with exact times
for bar, t in zip(bars, times):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f'{t:.3f}s',
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )

short_labels = ['Filter\n(F-stat)', 'Wrapper\n(RFE)', 'Embedded\n(L1 Logistic)']
ax.set_xticks(range(len(method_labels)))
ax.set_xticklabels(short_labels, fontsize=11)
ax.set_ylabel('Wall-clock time (seconds)', fontsize=12)
ax.set_title(
    f'Feature Selection Wall-Clock Time\n'
    f'Madelon dataset: p={X.shape[1]}, n={X.shape[0]}',
    fontsize=13
)

# Add complexity annotation
complexity_labels = ['O(p·n)', 'O(p·T_m)', 'O(T_m)']
for i, (bar, label) in enumerate(zip(bars, complexity_labels)):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        0.01,
        label,
        ha='center', va='bottom', fontsize=9, color='white', fontweight='bold'
    )

plt.tight_layout()
plt.savefig('timing_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nKey takeaway: timing ratios match the theoretical complexity formulas.')
print(f'Filter is ~{timing_results["Wrapper (RFE, step=10, k=20)"]["time"] / filter_time:.0f}x '
      f'faster than wrapper on p={X.shape[1]}.')

## Section 5: Do Selected Features Actually Help?

We have seen *how* methods differ in cost. Now let's verify that the selected features actually produce better models than using all 500 features.

We compare three pipelines:
1. All 500 features (baseline)
2. Top-20 by F-statistic filter
3. Features selected by L1 logistic regression (embedded)

All three use the same downstream classifier (logistic regression with L2 regularisation) and 5-fold CV.

In [ ]:
# Downstream classifier — same for all pipelines to isolate the effect of selection
downstream_clf = LogisticRegression(max_iter=300, random_state=42)
cv_splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


def evaluate_pipeline(X_feat, y, clf, cv):
    """5-fold CV accuracy and training time estimate."""
    t0 = time.perf_counter()
    scores = cross_val_score(clf, X_feat, y, cv=cv,
                              scoring='accuracy', n_jobs=-1)
    elapsed = time.perf_counter() - t0
    return scores.mean(), scores.std(), elapsed


# Build the three feature sets
# 1. All features
X_all = X_scaled

# 2. Top-20 filter
filter_selector = SelectKBest(f_classif, k=20)
X_filter = filter_selector.fit_transform(X_scaled, y)

# 3. Embedded L1 selection
lr_l1 = LogisticRegression(
    penalty='l1', solver='liblinear', C=0.1, max_iter=300, random_state=42
)
emb_selector = SelectFromModel(lr_l1, threshold=1e-5)
X_embedded = emb_selector.fit_transform(X_scaled, y)

pipelines = {
    f'All features ({X_all.shape[1]})': X_all,
    f'Filter top-20 ({X_filter.shape[1]})': X_filter,
    f'Embedded L1 ({X_embedded.shape[1]})': X_embedded,
}

print('Comparing downstream model accuracy after different selection strategies:')
print('=' * 65)

comparison_results = {}
for name, X_feat in pipelines.items():
    mean_acc, std_acc, t = evaluate_pipeline(X_feat, y, downstream_clf, cv_splitter)
    comparison_results[name] = {
        'accuracy': mean_acc,
        'std': std_acc,
        'n_features': X_feat.shape[1],
        'eval_time': t,
    }
    print(f'  {name}')
    print(f'    Accuracy: {mean_acc:.4f} ± {std_acc:.4f}')
    print(f'    CV time: {t:.2f}s')

In [ ]:
# Visualise accuracy comparison
fig, ax = plt.subplots(figsize=(9, 5))

names = list(comparison_results.keys())
accs = [comparison_results[n]['accuracy'] for n in names]
stds = [comparison_results[n]['std'] for n in names]

bar_colours = ['#9E9E9E', '#4CAF50', '#2196F3']  # grey=baseline, green=filter, blue=embedded

bars = ax.bar(
    range(len(names)), accs,
    color=bar_colours, alpha=0.85, edgecolor='white',
    linewidth=1.5, yerr=stds, capsize=5
)

for bar, acc, std in zip(bars, accs, stds):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        acc + std + 0.002,
        f'{acc:.4f}',
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )

ax.set_xticks(range(len(names)))
ax.set_xticklabels([n.replace(' (', '\n(') for n in names], fontsize=10)
ax.set_ylabel('5-fold CV Accuracy', fontsize=12)
ax.set_title(
    'Downstream Accuracy: All Features vs. Selected Features\n'
    'Same logistic regression classifier in all cases',
    fontsize=13
)

# Add baseline reference line
baseline_acc = comparison_results[names[0]]['accuracy']
ax.axhline(baseline_acc, color='black', linestyle=':', alpha=0.5,
           linewidth=1.2, label='All-features baseline')

# Set y-axis to show differences clearly
y_min = min(accs) - 0.03
y_max = max(accs) + 0.03
ax.set_ylim(y_min, y_max)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('selection_accuracy_comparison.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nSelection improves accuracy because noise features hurt generalisation.')

## Summary

### Key Takeaways

1. **The Hughes phenomenon is real:** On Madelon, accuracy peaks at far fewer than the available 500 features and declines as noise features are added. Using all features is rarely optimal.

2. **Distance concentration is a geometric fact:** As $d$ increases, the coefficient of variation of pairwise distances collapses toward zero. By $d = 200$, all points are approximately equidistant — distance-based algorithms lose their primary signal.

3. **The timing ratios match theory:** Filter is orders of magnitude faster than wrapper. Embedded (L1 logistic) is competitive in both speed and accuracy. The complexity formulas from the guide predict real-world performance.

4. **Selected features generalise better:** Both filter and embedded selection outperform the all-features baseline in downstream accuracy — by removing noise features that hurt the classifier.

### What's Next

- **Notebook 02:** Deep dive into the benchmark datasets we will use throughout the course. You will build baseline models and reusable data loading functions.
- **Module 01:** Statistical filter methods in depth — mutual information, ReliefF, and multivariate filters.

### Going Further

- Try replacing `LogisticRegression` with `RandomForestClassifier` as the downstream model — does the optimal $k$ change?
- Change the wrapper `step` parameter to 1 (one feature removed per step) and observe how runtime changes versus step=10.
- On your own dataset: compute $p/n$ and compare to the risk table in the complexity guide to assess which methods are appropriate.